# Volume 13, Chapter 5: Exponential sums over primes (Python companion)

Exercises first, solutions at the end. We watch $\sum\Lambda(n)e(n\alpha)$ peak on the major arcs and shrink on the minor arcs, test the equidistribution of $p\sqrt2$ modulo $1$, check the exact moments of Kloosterman sums and the Sato–Tate law, and compare Kloosterman with Salié. The computer science application: **pseudorandom number generators**, where linear recurrences fall into planes and the inversive generator, whose quality is measured by Kloosterman sums, does not. Requires `numpy` and `sympy`.

In [ ]:
import math, time
import numpy as np
from sympy import primerange, legendre_symbol

def mangoldt_array(N):
    lam = np.zeros(N + 1)
    for p in primerange(2, N + 1):
        pk = p
        while pk <= N:
            lam[pk] = math.log(p); pk *= p
    return lam

def kloosterman_all(p):
    # K(1,n;p) for n = 0..p-1 with one FFT: K(1,n) = sum_y e((n y + ybar)/p)
    inv = np.zeros(p, dtype=np.int64)
    inv[1:] = [pow(y, -1, p) for y in range(1, p)]
    f = np.exp(2j * np.pi * inv / p); f[0] = 0
    return np.real(np.fft.ifft(f) * p)

## Exercises

**P5.1 (★) Major and minor arcs.** For $N=10^6$, compute $S(\alpha)=\sum_{n\le N}\Lambda(n)e(n\alpha)$ at $\alpha=\frac aq$ for $q\le 12$ and compare $S/\psi(N)$ with $\frac{\mu(q)}{\varphi(q)}$ (Lemma 5.4). Then compute $|S(\alpha)|/\psi(N)$ for $\alpha=\sqrt2$, $\frac{1+\sqrt5}2$, $e$ and $\pi$ at $N=10^4,10^5,10^6$. How fast does it decrease? Why is $\pi$ different?

**P5.2 (★) Equidistribution of $p\sqrt2$.** Compute the discrepancy $D_N=\sup_{0\le t\le1}\bigl|\frac1{\pi(N)}\#\{p\le N:\{p\sqrt2\}<t\}-t\bigr|$ for $N=10^3,\dots,10^6$, and the same for $\{n\sqrt2\}$, $n\le\pi(N)$. Which converges faster?

**P5.3 (★★) Kloosterman against Salié.** For $p=10007$, compute all $K(1,n;p)$ with one FFT. Check Weil's bound, and check that $\sum_{n=1}^{p-1}K^2$ and $\sum K^4$ are the integers of Lemma 5.9. Compare the histogram of $K/(2\sqrt p)$ with the semicircle, and that of the Salié sums $S(1,n;p)/(2\sqrt p)$, $n$ a residue, with the arcsine density $\frac1{\pi\sqrt{1-x^2}}$ (up to sign). Check Theorem 5.12 for a few $n$.

**P5.4 (★★) Pseudorandom generators (computer science).** (a) IBM's RANDU, $x_{n+1}=65539x_n\bmod2^{31}$, was the standard generator of the 1960s. Show that $x_{n+2}-6x_{n+1}+9x_n\equiv0\pmod{2^{31}}$, and deduce that the triples $(x_n,x_{n+1},x_{n+2})/2^{31}$ lie on at most $15$ planes. Check it numerically. (b) Compare RANDU with the inversive generator $x_{n+1}=a\overline{x_n}+b\bmod p$, $p=2^{31}-1$, in a three-dimensional serial test: the $\chi^2$ statistic of $10^6$ triples in $16^3$ boxes. (c) For the inversive generator modulo a small prime $p=1009$, count the points $(x,\overline x)$ in all boxes $[1,u]\times[1,v]$ and compare the largest error with $p^{3/4}\log^2p$ and $\sqrt p\log^2p$ (Exercise 5.6).

## Solutions

### P5.1
One pass over the $\Lambda(n)$ for each $\alpha$; no FFT needed.

In [ ]:
N = 10**6
lam = mangoldt_array(N); n = np.arange(N + 1); psiN = lam.sum()
def S(alpha, M=N):
    return np.sum(lam[:M+1] * np.exp(2j * np.pi * (n[:M+1] * alpha % 1.0)))
from sympy import mobius, totient
for q in range(1, 13):
    for a in range(1, q + 1):
        if math.gcd(a, q) == 1 and 2 * a <= q + 1:
            v = S(a / q)
            print(f"a/q = {a}/{q:2d}:  S/psi = {v.real/psiN:+.4f}{v.imag/psiN:+.4f}i   mu/phi = {int(mobius(q))/int(totient(q)):+.4f}")
            break
for name, al in [("sqrt2", math.sqrt(2)), ("golden", (1 + math.sqrt(5)) / 2), ("e", math.e), ("pi", math.pi)]:
    print(name, [round(abs(S(al, M)) / lam[:M+1].sum(), 5) for M in (10**4, 10**5, 10**6)])

**What it shows.** At $\frac aq$ (one $a$ per $q$ is printed) the ratio $S/\psi(N)$ matches $\frac{\mu(q)}{\varphi(q)}$ to about three decimals: $-1$ at $q=2$, $-\frac12$ at $q=3$ and $6$ (real part), $0$ at $q=4,8,9,12$, $-\frac14$ at $q=5$, $\frac14$ at $q=10$, $-\frac16$ at $q=7$, $-\frac1{10}$ at $q=11$; the imaginary parts are $O(10^{-3})$. For the irrationals, $|S|/\psi(N)$ is between $0.003$ and $0.017$ at $N=10^4$ and mostly decreases, for $\sqrt2$ by a factor of about $4$ per decade ($0.013$, $0.0036$, $0.0008$), far below what Theorem 5.3 guarantees. The decrease is irregular, because it depends on how close $\alpha$ is, at the scale $N$, to fractions with moderate denominators. The number $\pi$ shows this best: $|\pi-\frac{355}{113}|=\beta=2.7\cdot10^{-7}$, so for $N\le10^6$, $\pi$ sits on the major arc of $\frac{355}{113}$, and Lemma 5.4 predicts $|S|/\psi(N)\approx\frac1{\varphi(113)}\bigl|\frac{\sin\pi\beta N}{\pi\beta N}\bigr|$, which is $0.0079$ at $N=10^6$ (where $\beta N=0.27$); the computed value is $0.0078$. Only for $N$ well beyond $\frac1\beta\approx4\cdot10^6$ does the factor $e(\beta n)$ make the sum cancel.

### P5.2
Sort the fractional parts; the discrepancy of a sorted sample $u_{(1)}\le\dots\le u_{(m)}$ is $\max_i\max(\frac im-u_{(i)},u_{(i)}-\frac{i-1}m)$.

In [ ]:
def disc(u):
    u = np.sort(u); m = len(u); i = np.arange(1, m + 1)
    return max((i / m - u).max(), (u - (i - 1) / m).max())
P = np.array(list(primerange(2, 10**6 + 1)))
r2 = math.sqrt(2)
for M in (10**3, 10**4, 10**5, 10**6):
    pm = P[P <= M]; k = len(pm)
    print(f"N = 10^{round(math.log10(M))}: pi(N) = {k:6d}   D(p sqrt2) = {disc((pm * r2) % 1):.5f}   D(n sqrt2, n <= pi(N)) = {disc((np.arange(1, k + 1) * r2) % 1):.5f}")

**What it shows.** Both discrepancies tend to $0$. For $n\sqrt2$ the decay is spectacular, about $\frac{\log m}m$ ($0.0077$ for $m=168$ points, $3\cdot10^{-5}$ for $m=78498$), because the continued fraction of $\sqrt2$ has bounded partial quotients. For the primes it is about $0.4/\sqrt{\pi(N)}$ ($0.069$, $0.013$, $0.0045$, $0.0015$): the rate of independent random points, for which the mean discrepancy is about $0.87/\sqrt m$. So the primes, seen through the rotation by $\sqrt2$, look like random points, while the integers look far more regular than random. Vinogradov's theorem only guarantees $D_N\to0$.

### P5.3
`kloosterman_all` computes the $p$ sums in $O(p\log p)$ operations; the Salié sums come from Theorem 5.12, and we check a few against the definition.

In [ ]:
p = 10007
K = kloosterman_all(p)
x = K[1:] / (2 * math.sqrt(p))
print("max |K|/(2 sqrt p) =", round(float(np.abs(x).max()), 5))
print("sum K^2 =", round(float((K[1:]**2).sum())), " p^2-p-1 =", p*p - p - 1)
print("sum K^4 =", round(float((K[1:]**4).sum())), " 2p^3-3p^2-3p-1 =", 2*p**3 - 3*p*p - 3*p - 1)
h, _ = np.histogram(x, bins=10, range=(-1, 1))
semi = [(2/math.pi) * (math.asin(b) + b*math.sqrt(1-b*b)) / 2 for b in np.linspace(-1, 1, 11)]
print("Kloosterman bins:", np.round(h / (p - 1), 3))
print("semicircle     :", np.round(np.diff(semi), 3))
# Salie sums via Theorem 5.12, checked against the definition for a few n
g = sum(legendre_symbol(t, p) * np.exp(2j * np.pi * t / p) for t in range(1, p))
def salie_formula(m, n):
    ys = [y for y in range(p) if (y * y - m * n) % p == 0]
    return legendre_symbol(n, p) * g * sum(np.exp(4j * np.pi * y / p) for y in ys)
def salie_direct(m, n):
    return sum(legendre_symbol(t, p) * np.exp(2j * np.pi * ((m * t + n * pow(t, -1, p)) % p) / p) for t in range(1, p))
for nn in (1, 2, 5):
    print(f"Salie S(1,{nn}): formula {complex(salie_formula(1, nn)):.4f}  direct {complex(salie_direct(1, nn)):.4f}")
sq = {(y * y) % p: y for y in range(1, p)}
vals = np.array([math.cos(4 * math.pi * sq[nn] / p) for nn in range(1, p) if nn in sq])
arcs = [(math.asin(b) / math.pi) for b in np.linspace(-1, 1, 11)]
print("Salie |cos| bins :", np.round(np.histogram(vals, bins=10, range=(-1, 1))[0] / len(vals), 3))
print("arcsine          :", np.round(np.diff(arcs), 3))

**What it shows.** For $p=10007$ the largest $|K|$ is $0.9994\cdot2\sqrt p$: Weil's bound holds and is nearly attained. The two moments are exactly the integers $p^2-p-1=100130041$ and $2p^3-3p^2-3p-1=2003902490517$ of Lemma 5.9. The Kloosterman histogram follows the semicircle bin by bin, to within about $0.004$: Katz's law is visible already at one prime. The Salié formula of Theorem 5.12 agrees with the definition; here $p\equiv3\pmod4$, so $g_p=\pm i\sqrt p$ and the Salié sums are purely imaginary, as printed. The cosines $\cos\frac{4\pi y}p$ that make up the Salié sums follow the arcsine law exactly, with mass piled up near $\pm1$, not the semicircle: two families of sums of the same shape, with different statistics.

### P5.4
For RANDU, $65539=2^{16}+3$, so $65539^2=2^{32}+6\cdot2^{16}+9\equiv6\cdot65539-9\pmod{2^{31}}$.

In [ ]:
M31 = 2**31
x = 1; seq = []
for _ in range(3 * 10**5 + 2):
    x = 65539 * x % M31; seq.append(x)
seq = np.array(seq, dtype=np.int64)
rel = (seq[2:] - 6 * seq[1:-1] + 9 * seq[:-2]) % M31
print("RANDU: x_{n+2} - 6 x_{n+1} + 9 x_n = 0 mod 2^31 for all n:", bool((rel == 0).all()))
u = seq / M31
planes = np.round(u[2:] - 6 * u[1:-1] + 9 * u[:-2]).astype(int)
print("values of the integer k in u_{n+2} - 6u_{n+1} + 9u_n = k:", sorted(set(planes)), "->", len(set(planes)), "planes")

def chi2_triples(u, m=16):
    t = (u * m).astype(int).clip(0, m - 1)
    idx = t[:-2:3] * m * m + t[1:-1:3] * m + t[2::3]
    c = np.bincount(idx, minlength=m**3); e = len(idx) / m**3
    return float(((c - e)**2 / e).sum()), m**3 - 1

def randu(n, seed=1):
    out = np.empty(n); x = seed
    for i in range(n):
        x = 65539 * x % M31; out[i] = x / M31
    return out
def inversive(n, p=2**31 - 1, a=1288490188, b=1, seed=1):
    out = np.empty(n); x = seed
    for i in range(n):
        x = (a * pow(x, -1, p) + b) % p if x else b; out[i] = x / p
    return out
t0 = time.time()
for name, gen in (("RANDU", randu), ("inversive", inversive)):
    stat, df = chi2_triples(gen(3 * 10**6))
    print(f"{name:10s} chi^2 = {stat:12.1f} on {df} degrees of freedom (z = {(stat - df) / math.sqrt(2 * df):+.1f})")
print(f"({time.time() - t0:.0f}s)")

p = 1009
inv = np.zeros(p + 1, dtype=np.int64); inv[1:p] = [pow(t, -1, p) for t in range(1, p)]
grid = np.zeros((p, p), dtype=np.int64)
for t in range(1, p):
    grid[t, inv[t]] = 1
cum = grid.cumsum(0).cumsum(1)
uu, vv = np.meshgrid(np.arange(p), np.arange(p), indexing="ij")
err = np.abs(cum - uu * vv / p).max()
print(f"p = {p}: largest box error {err:.1f};  p^(3/4) log^2 p = {p**0.75 * math.log(p)**2:.0f};  sqrt(p) log^2 p = {math.sqrt(p) * math.log(p)**2:.0f}")

**What it shows.** (a) The relation holds for every $n$, and $u_{n+2}-6u_{n+1}+9u_n$ is an integer $k$ between $-5$ and $9$, so at most $15$ values: every RANDU triple lies on one of $15$ parallel planes in the unit cube (Marsaglia's observation). (b) In the serial test with $4096$ boxes, RANDU's $\chi^2$ is enormous, about $380000$, some $4000$ standard deviations above its expectation, because most boxes meet none of the planes; the inversive generator's statistic, about $4009$, is one standard deviation from $4095$, as for truly random triples. (c) For $p=1009$ the largest error in counting $(x,\overline x)$ in a box anchored at the origin is about $20$, against $p^{3/4}\log^2p\approx8600$ and $\sqrt p\log^2p\approx1500$: both bounds hold with room to spare, and the truth is below $\sqrt p\approx32$, as Weil's bound suggests. The price of the inversive generator is one modular inversion per number, about $\log p$ multiplications.

*References.* G. Marsaglia, "Random numbers fall mainly in the planes", *Proc. Nat. Acad. Sci. USA* 61 (1968), 25–28. J. Eichenauer and J. Lehn, "A non-linear congruential pseudo random number generator", *Statist. Hefte* 27 (1986), 315–326. D. E. Knuth, *The Art of Computer Programming*, Vol. 2, 3rd ed., Addison-Wesley, 1997, Section 3.3.4 (the spectral test).